# UC-PROC-2 — Long-Running Async Job

**Persona:** Data Engineer

**Goal:** Trigger a MODIS reprocessing job with `Prefer: respond-async`, poll through
`accepted → running → successful` using exponential backoff, retrieve the result
documents, and dismiss the job.

**Prerequisites:**
- OGC API Processes extension registered; catalog-scoped process route enabled
- Process `PROCESS_ID` registered for the target catalog and able to accept the
  reprocess inputs below
- Write-capable JWT in `DYNASTORE_WRITE_TOKEN`

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_WRITE_TOKEN`, `CATALOG_ID`,
`PROCESS_ID`

In [ ]:
import os
import json
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["DYNASTORE_BASE_URL"]
WRITE_TOKEN = os.environ["DYNASTORE_WRITE_TOKEN"]
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
PROCESS_ID = os.environ["PROCESS_ID"]

headers = {
    "Authorization": f"Bearer {WRITE_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

job_id = None

print(f"Connected to {BASE_URL}")
print(f"CATALOG_ID={CATALOG_ID}  PROCESS_ID={PROCESS_ID}")

In [ ]:
# Trigger the reprocessing job asynchronously
execution_payload = {
    "inputs": {
        "product": "MODIS_MOD13Q1",
        "date_range": {"start": "2024-01-01", "end": "2024-03-31"},
        "bands": ["NDVI", "EVI"],
        "output_format": "GeoTIFF",
    },
    "outputs": {"result_collection": {"transmissionMode": "reference"}},
}

r = client.post(
    f"/processes/catalogs/{CATALOG_ID}/processes/{PROCESS_ID}/execution",
    content=json.dumps(execution_payload),
    headers={"Prefer": "respond-async"},
)
print(r.status_code, r.text[:400])
assert r.status_code == 202, (
    f"Expected 202 (async accepted), got {r.status_code}: {r.text}"
)

receipt = r.json()
status = receipt.get("status", "")
assert status == "accepted", (
    f"Expected status='accepted' in 202 response, got '{status}': {receipt}"
)

job_id = receipt.get("jobID", receipt.get("job_id", ""))
assert job_id, f"No jobID in 202 response: {receipt}"
print(f"Job accepted: job_id={job_id}")

In [ ]:
# Poll with exponential backoff until terminal status
status = "accepted"
for attempt in range(20):
    resp = client.get(f"/processes/jobs/{job_id}")
    assert resp.status_code == 200, (
        f"Expected 200 polling job, got {resp.status_code}: {resp.text}"
    )
    status = resp.json().get("status", "")
    print(f"Attempt {attempt + 1}: status={status}")
    if status in ("successful", "failed", "dismissed"):
        break
    time.sleep(min(2 ** attempt, 30))

assert status in ("successful", "failed"), (
    f"Job did not reach a terminal status after 20 attempts; last status='{status}'"
)
print(f"Job reached terminal status: {status}")

In [ ]:
# Verify the job appears in the catalog-scoped job list
r = client.get(f"/processes/catalogs/{CATALOG_ID}/jobs")
print(r.status_code)
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

body = r.json()
jobs = body.get("jobs", body) if isinstance(body, dict) else body
if isinstance(jobs, dict):
    jobs = list(jobs.values())

job_ids_in_list = [
    j.get("jobID", j.get("job_id", j.get("id", ""))) for j in jobs
]
print(f"Jobs in catalog scope ({len(job_ids_in_list)}): {job_ids_in_list[:10]}")
assert job_id in job_ids_in_list, (
    f"job_id '{job_id}' not found in catalog job list: {job_ids_in_list}"
)
print(f"Confirmed: job {job_id} is listed under catalog {CATALOG_ID}.")

In [ ]:
# Retrieve results (only if job completed successfully)
if status == "successful":
    r = client.get(f"/processes/jobs/{job_id}/results")
    print(r.status_code)
    assert r.status_code == 200, f"Expected 200 fetching results, got {r.status_code}: {r.text}"
    results = r.json()
    print("Results:")
    print(json.dumps(results, indent=2)[:1000])
else:
    print(f"Job status='{status}'; skipping results fetch (see error-handling edge case below).")

In [ ]:
# Dismiss the job
r = client.delete(f"/processes/jobs/{job_id}")
print(r.status_code)
assert r.status_code in (200, 204), (
    f"Expected 200 or 204 dismissing job, got {r.status_code}: {r.text}"
)
print(f"Job {job_id} dismissed.")

## Edge Cases

In [ ]:
# Edge case 1: job status == 'failed' -> results contain exception detail
#
# When a job fails, GET /processes/jobs/{job_id}/results returns 200 with a
# structured OGC exception report rather than output values:
#
#   {
#     "type": "http://www.opengis.net/def/exceptions/ogcapi-processes-1/1.0/job-result-failed",
#     "title": "Job result failed",
#     "detail": "<error message from the process worker>",
#     "instance": "/processes/jobs/{job_id}/results"
#   }
#
# Robust handling pattern:
def handle_job_results(client: httpx.Client, job_id: str) -> dict:
    r = client.get(f"/processes/jobs/{job_id}/results")
    assert r.status_code == 200, f"Unexpected status {r.status_code}: {r.text}"
    body = r.json()
    if "type" in body and "exception" in body.get("type", "").lower():
        raise RuntimeError(
            f"Job {job_id} failed: {body.get('detail', body)}"
        )
    return body

print("handle_job_results() defined: raises RuntimeError if job result is an exception report.")

In [ ]:
# Edge case 2: job not owned by caller -> 404 (tenant isolation)
#
# Jobs are scoped to the submitting tenant. If a different tenant (or an unauthenticated
# caller) attempts to GET a job belonging to another tenant, the platform returns 404
# rather than 403 to avoid leaking the existence of the job.
#
# Demonstrate with a fabricated job_id:
fake_job_id = "00000000-0000-0000-0000-000000000000"
r = client.get(f"/processes/jobs/{fake_job_id}")
print(r.status_code)
assert r.status_code == 404, (
    f"Expected 404 for non-existent/foreign job, got {r.status_code}: {r.text}"
)
print("404 confirmed: cross-tenant job access correctly denied without leaking existence.")

In [ ]:
# Edge case 3: Cloud Run timeout warning for jobs exceeding 15 minutes
#
# Cloud Run HTTP requests time out at 60 minutes (default 5 minutes without override).
# For long-running async jobs the worker is a background task; the HTTP response (202)
# is returned immediately. However, if the compute backend runs on the same Cloud Run
# instance, the worker is bounded by the instance lifecycle.
#
# Jobs expected to run for > 15 minutes MUST be dispatched to Cloud Run Jobs
# (not Cloud Run Services) to avoid premature termination on instance scale-down.
#
# Detection: if a job transitions from 'running' directly to 'failed' with
# detail containing 'signal: killed' or 'worker timeout', this is the symptom.
#
# Mitigation checklist:
#   1. Register the process with `execution_unit: cloud_run_job` in the process descriptor.
#   2. Set Cloud Run Job max-retries=0 if the process is not idempotent.
#   3. Ensure the job emits heartbeat status updates so the poll loop does not
#      time out on the client side.
print(
    "Cloud Run timeout guidance: jobs > 15 min require Cloud Run Job mode.\n"
    "Register process with execution_unit='cloud_run_job' to avoid instance kill."
)
client.close()